# 04. 모델링 (Modeling)

**모델 스택**: Logistic Regression (베이스라인) → LightGBM → XGBoost  
**검증 방법**: Stratified K-Fold (k=5), SMOTE는 각 fold 내부에서 적용  
**주 평가 지표**: AUC-ROC  
**부 평가 지표**: F1-score, Precision, Recall

---
## Step 0. 라이브러리 & 설정

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, roc_curve
)
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_theme(font_scale=1.5)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs', exist_ok=True)
print('라이브러리 로드 완료')

---
## Step 1. 데이터 로드

In [ ]:
X_train = pd.read_parquet('../data/interim/X_train.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')['label']
X_test  = pd.read_parquet('../data/interim/X_test.parquet')
y_test  = pd.read_parquet('../data/interim/y_test.parquet')['label']

feature_cols = X_train.columns.tolist()

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'피처: {feature_cols}')
print(f'\nTrain 클래스 분포: {y_train.value_counts().to_dict()}')
print(f'Test  클래스 분포: {y_test.value_counts().to_dict()}')

---
## Step 2. CV 헬퍼 & SKF 설정

- SMOTE는 각 fold의 train split에만 적용 (pipeline 내부에서 처리)
- `predict_proba`가 없는 모델 대비 `decision_function` fallback 불필요 — 세 모델 모두 지원

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(pipeline, X, y, model_name):
    """Stratified K-Fold CV. pipeline에 SMOTE 포함 가정."""
    results = {'auc': [], 'f1': [], 'precision': [], 'recall': []}
    print(f'=== {model_name} — CV (k=5) ===')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        pipeline.fit(X_tr, y_tr)
        y_prob = pipeline.predict_proba(X_val)[:, 1]
        y_pred = pipeline.predict(X_val)
        results['auc'].append(roc_auc_score(y_val, y_prob))
        results['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        results['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        results['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        print(f'  Fold {fold}: AUC={results["auc"][-1]:.4f}  F1={results["f1"][-1]:.4f}')
    print(f'  → CV AUC: {np.mean(results["auc"]):.4f} ± {np.std(results["auc"]):.4f}\n')
    return results


def eval_test(pipeline, X_test, y_test):
    """Test set 최종 평가."""
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    return {
        'test_auc'      : roc_auc_score(y_test, y_prob),
        'test_f1'       : f1_score(y_test, y_pred, zero_division=0),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall'   : recall_score(y_test, y_pred, zero_division=0),
        'y_prob'        : y_prob,
    }

print('헬퍼 함수 정의 완료')

---
## Step 3. Logistic Regression (베이스라인)

- Pipeline: SMOTE → StandardScaler → LogisticRegression
- `class_weight='balanced'` 대신 SMOTE로 불균형 처리 (pipeline 통일)

In [ ]:
lr_pipe = ImbPipeline([
    ('smote',  SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42)),
])

lr_cv = run_cv(lr_pipe, X_train, y_train, 'Logistic Regression')

In [ ]:
# 전체 train으로 최종 학습 후 test 평가
lr_pipe.fit(X_train, y_train)
lr_test = eval_test(lr_pipe, X_test, y_test)

print('=== Logistic Regression — Test 결과 ===')
for k, v in lr_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

---
## Step 4. LightGBM

In [ ]:
lgb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lgb',   lgb.LGBMClassifier(n_estimators=300, random_state=42,
                                  n_jobs=-1, verbose=-1)),
])

lgb_cv = run_cv(lgb_pipe, X_train, y_train, 'LightGBM')

In [ ]:
lgb_pipe.fit(X_train, y_train)
lgb_test = eval_test(lgb_pipe, X_test, y_test)

print('=== LightGBM — Test 결과 ===')
for k, v in lgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

---
## Step 5. XGBoost

In [ ]:
xgb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb',   xgb.XGBClassifier(n_estimators=300, random_state=42,
                                  n_jobs=-1, verbosity=0, eval_metric='auc')),
])

xgb_cv = run_cv(xgb_pipe, X_train, y_train, 'XGBoost')

In [ ]:
xgb_pipe.fit(X_train, y_train)
xgb_test = eval_test(xgb_pipe, X_test, y_test)

print('=== XGBoost — Test 결과 ===')
for k, v in xgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

---
## Step 6. 모델 비교

In [ ]:
models = {
    'Logistic Regression': (lr_cv, lr_test),
    'LightGBM'           : (lgb_cv, lgb_test),
    'XGBoost'            : (xgb_cv, xgb_test),
}

rows = []
for name, (cv, test) in models.items():
    rows.append({
        '모델'                     : name,
        'CV AUC (mean)'            : f"{np.mean(cv['auc']):.4f}",
        'CV AUC (std)'             : f"{np.std(cv['auc']):.4f}",
        'Test AUC'                 : f"{test['test_auc']:.4f}",
        'Test F1'                  : f"{test['test_f1']:.4f}",
        'Test Precision'           : f"{test['test_precision']:.4f}",
        'Test Recall'              : f"{test['test_recall']:.4f}",
    })

comp_df = pd.DataFrame(rows).set_index('모델')
print('=== 모델 비교 ===')
print(comp_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = {'Logistic Regression': 'steelblue', 'LightGBM': 'tomato', 'XGBoost': 'seagreen'}
for name, (_, test) in models.items():
    fpr, tpr, _ = roc_curve(y_test, test['y_prob'])
    auc = test['test_auc']
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})", color=colors[name])

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Test Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('[저장] ../outputs/roc_curves.png')

---
## Step 7. 피처 중요도

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# LR: 계수 절댓값
lr_coef = np.abs(lr_pipe.named_steps['lr'].coef_[0])
# LGB / XGB: feature_importances_ (gain 기준)
lgb_imp = lgb_pipe.named_steps['lgb'].feature_importances_
xgb_imp = xgb_pipe.named_steps['xgb'].feature_importances_

for ax, imp, title in zip(
    axes,
    [lr_coef, lgb_imp, xgb_imp],
    ['LR (|coef|)', 'LightGBM', 'XGBoost']
):
    order = np.argsort(imp)
    ax.barh([feature_cols[i] for i in order], imp[order])
    ax.set_title(title)
    ax.set_xlabel('중요도')

plt.suptitle('피처 중요도 비교', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('[저장] ../outputs/feature_importance.png')

---
## Step 8. 결과 저장

In [ ]:
# 모델 비교 테이블 저장
comp_df.to_csv('../outputs/model_comparison.csv')
print('[저장] ../outputs/model_comparison.csv')

print('\n=== 최종 모델 비교 ===')
print(comp_df.to_string())